# Secure debug enablement on AHABv1 based devices
This notebook describes how to enable secure debug on AHAB based i.MX devices using the Lauterbach debugger and SPSDK.

## 1. Prerequisites
- SPSDK is needed with examples extension. `pip install spsdk[examples]` (refer to the [installation](../../_knowledge_base/installation_guide.rst) documentation)
- The device must be properly configured to run a signed bootable AHAB image (example for AHAB based devices: [AHAB Secure boot](../../ahab/imx93/imx93_signed_ahab_uboot.ipynb))
- The device lifecycle must be set to OEM_CLOSED
- SPSDK Lauterbach plugin: `pip install spsdk-lauterbach`
- This demo was tested with i.MX93 EVK board. External Lauterbach debugger was connected over the standard 5-pins JTAG connector

## 2. Secure debug enablement

Secure debug can only be enabled when the device is in OEM_CLOSED lifecycle. In this lifecycle, only authenticated debug is allowed. If needed, the debugging capability can be disabled with fuse programming. For more details, refer to the Processor Security Reference Manual.

### 2.1 Prepare the environment

The process will be demonstrated on an i.MX93 device; however, the same steps apply to all AHABv1-based devices.

In [1]:
from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


In [2]:
WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"
VERBOSITY = (
    "-v"  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)

DAT_CONFIG = INPUTS + "dat_config.yaml"  # yaml file with configuration used in this example
DC_CONFIG = (
    INPUTS + "dc_config.yaml"
)  # yaml file with configuration used in this example for debug credential file

DEBUGGER_PROBE = "lauterbach"
# Only one probe can be connected to the board.

FAMILY = "mimx9352"

### 2.2 Trace32 configuration

Once the tools are installed, the following change needs to be applied in the Trace32 config file `config.t32`, so that remote control is enabled.

```bash
<mandatory empty line>
; Remote Control Access
PORT=20000
<mandatory empty line>
```

Connect the Lauterbach debugger to the i.MX EVK JTAG, start Trace32 and use the debug startup script to attach to one of the cores (e.g. Cortex A55).


### 2.3 Signing keys

If a technician needs to enable this procedure and does not use the SRKs, the key pair required to sign the DC can be created using nxpcrypto. The command below generates a secp384 key pair. For other key types or additional details, refer to the [nxpcrypto documentation](https://spsdk.readthedocs.io/en/latest/usage/nxpcrypto.html).


In [3]:
DCK = INPUTS + "dck_ecc384.pem"
%! nxpcrypto key generate -k secp384r1 -o $DCK -e pem --force

nxpcrypto key generate -k secp384r1 -o inputs/dck_ecc384.pem -e pem --force 
The key pair has been created: inputs/dck_ecc384.pub, inputs/dck_ecc384.pem


### 2.4 UUID retrieval

To enable secure debug for a specific part, its UUID must be specified in the SPSDK DC configuration template. The device UUID can be retrieved using `nxpdebugmbox`:

In [4]:

%! nxpdebugmbox -i $DEBUGGER_PROBE tool get-uuid -f $FAMILY

nxpdebugmbox -i lauterbach tool get-uuid -f mimx9352 
  #   Interface    Id             Description                           Architecture   Protocols  
--------------------------------------------------------------------------------------------------
  0   lauterbach   C18050250094   Lauterbach probe at localhost:20000   arm-cortex     swd        
The device UUID is: 8b20c04116884f55a9bc726d290670de


### 2.5 Debug credential

The template configuration file can be created using `nxpdebugmbox dat dc get-template`. To simplify this example, we have already prepared that configuration, which can be found in the ./inputs/dc_config.yaml. Below, we'll compare the differences between the template and our customized example to highlight the additions we've made.

In [5]:
# Get difference of template and user YAML configuration
YamlDiffWidget("inputs/dc_config.diffc").html

nxpdebugmbox dat dc get-template -f mimx9352 -o workspace/dc_config.yaml --force 
Creating workspace/dc_config.yaml template file.


For additional information regarding the fields consult the Processor Security Reference Manual and the EdgeLock Enclave API Reference Guide.

### 2.6 Export the DC

In [6]:

DC_FILE_PATH = WORKSPACE + "imx93.dc"
%! nxpdebugmbox dat dc export -c $DC_CONFIG -o $DC_FILE_PATH --force

nxpdebugmbox dat dc export -c inputs/dc_config.yaml -o workspace/imx93.dc --force 
RKTH: b19c214ea5b4217008c51a12f6e1846c75b75a6e7052e9b454804fdae8d09f61
Creating Debug credential file succeeded


### 2.7 DAT

The template configuration file can be created using `nxpdebugmbox dat get-template`. To simplify this example, we have already prepared that configuration, which can be found in the ./inputs/dat_config.yaml. Below, we'll compare the differences between the template and our customized example to highlight the additions we've made.

In [7]:
# Get difference of template and user YAML configuration
YamlDiffWidget("inputs/dat_config.diffc").html

nxpdebugmbox dat get-template -f mimx9352 -o workspace/dat_config.yaml --force 
Creating workspace/dat_config.yaml template file.


### 2.8 Perform the DAT authentication

In [8]:
# Unlock the Debug Authentication
%! nxpdebugmbox -i $DEBUGGER_PROBE dat auth -c $DAT_CONFIG

nxpdebugmbox -i lauterbach dat auth -c inputs/dat_config.yaml 
  #   Interface    Id             Description                           Architecture   Protocols  
--------------------------------------------------------------------------------------------------
  0   lauterbach   C18050250094   Lauterbach probe at localhost:20000   arm-cortex     swd        
Testing AHB access (Press Ctrl+C to stop)
Debug Authentication ends successfully.


### 2.9 Test the connection

Check the connection availability using `nxpdebugmbox`.

**Note:** When the device is in the OEM closed lifecycle state, the command will fail unless a valid debug authentication is provided.

In case you encounter issues with the connection try to power off the i.MX device, unplug the debugger, reconnect it and power on the device.

In [9]:
%! nxpdebugmbox -i $DEBUGGER_PROBE mem-tool test-connection -f $FAMILY

nxpdebugmbox -i lauterbach mem-tool test-connection -f mimx9352 
  #   Interface    Id             Description                           Architecture   Protocols  
--------------------------------------------------------------------------------------------------
  0   lauterbach   C18050250094   Lauterbach probe at localhost:20000   arm-cortex     swd        
The test connection ends successfully.


`nxpdebugmbox mem-tool` can be used to perform read and write operations once debug has been enabled, as illustrated below:

In [10]:
%! nxpdebugmbox -i $DEBUGGER_PROBE mem-tool read-memory -a 0x20480000 -c 20 -f $FAMILY

nxpdebugmbox -i lauterbach mem-tool read-memory -a 0x20480000 -c 20 -f mimx9352 
  #   Interface    Id             Description                           Architecture   Protocols  
--------------------------------------------------------------------------------------------------
  0   lauterbach   C18050250094   Lauterbach probe at localhost:20000   arm-cortex     swd        
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
00 00 00 00


In [11]:
%! nxpdebugmbox -i $DEBUGGER_PROBE mem-tool write-memory -a 0x20480000 -h 1234 -c 20 -f $FAMILY

nxpdebugmbox -i lauterbach mem-tool write-memory -a 0x20480000 -h 1234 -c 20 -f mimx9352 
  #   Interface    Id             Description                           Architecture   Protocols  
--------------------------------------------------------------------------------------------------
  0   lauterbach   C18050250094   Lauterbach probe at localhost:20000   arm-cortex     swd        
The memory has been written successfully.


In [12]:
%! nxpdebugmbox -i $DEBUGGER_PROBE mem-tool read-memory -a 0x20480000 -c 20 -f $FAMILY

nxpdebugmbox -i lauterbach mem-tool read-memory -a 0x20480000 -c 20 -f mimx9352 
  #   Interface    Id             Description                           Architecture   Protocols  
--------------------------------------------------------------------------------------------------
  0   lauterbach   C18050250094   Lauterbach probe at localhost:20000   arm-cortex     swd        
12 34 00 00 00 00 00 00 00 00 00 00 00 00 00 00
00 00 00 00
